## Aim:

This notebook takes a directory of DEM tiles (imported from Copernicus Data Explorer, or DEFRA (Uk.gov)), and exports a merged DEM .tif.

AI helped, but did NOT generate, parts of the logic in this code. Specifically, it helped with the tempfile functionality, as this was new to me.

In [2]:
import rasterio
from rasterio.merge import merge
from pathlib import Path
import numpy as np

dem_files = sorted(Path('../../data/Quantock/RawDEMDir').glob('*.tif')) 
OUT_PATH = Path('../../data/Quantock/out/')

In [ ]:
import tempfile
import os

clipped_paths = []
tmpdir = tempfile.mkdtemp()

for f in dem_files:
    with rasterio.open(f) as src:
        data = src.read(1)
        # clips sealevel and NODATA values (1e-38) to 0.
        data = np.clip(data, 0, None).astype(np.float32)
        
        profile = src.profile.copy()
        profile.update(dtype='float32', nodata=0)
        
        out_path = os.path.join(tmpdir, f.name)
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data, 1)
        
        clipped_paths.append(out_path)
        print(f"{f.name} — min {data[data>0].min():.2f}  max {data.max():.2f}")

# now merge the clipped files
datasets = [rasterio.open(p) for p in clipped_paths]
mosaic, transform = merge(datasets)

print(f"Mosaic — min {mosaic.min():.2f}  max {mosaic.max():.2f}")

ST12ne_DTM_1m.tif — min 17.39  max 97.41
ST12nw_DTM_1m.tif — min 39.42  max 178.23
ST13nw_DTM_1m.tif — min 39.15  max 358.09
ST13se_DTM_1m.tif — min 50.14  max 377.81
ST13sw_DTM_1m.tif — min 73.07  max 218.80
Mosaic — min 0.00  max 377.81


In [ ]:
profile = datasets[0].profile
profile.update(
    width=mosaic.shape[2],
    height=mosaic.shape[1],
    transform=transform,
    dtype='float32',
    nodata=0,
)

with rasterio.open(OUT_PATH / 'merged_dem.tif', 'w', **profile) as dst:
    dst.write(mosaic)

for ds in datasets:
    ds.close()

print(f"Saved")


ERROR 10: Pointer 'hObject' is NULL in 'GDALGetMetadata'.

ERROR 10: Pointer 'hObject' is NULL in 'GDALGetMetadata'.

ERROR 10: Pointer 'hObject' is NULL in 'GDALGetMetadata'.



Saved
